In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
df=pd.read_csv('/content/fashion-mnist_train.csv')

In [ ]:
print(df.shape)

(60000, 785)


In [ ]:
df.sample(5)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
10865,2,0,0,0,0,0,0,2,52,98,...,0,0,0,0,66,162,75,0,0,0
41327,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59265,2,0,0,0,0,1,0,0,0,0,...,0,0,10,75,70,9,0,0,0,0
4196,4,0,0,0,0,0,0,0,4,0,...,179,177,143,0,0,0,0,0,0,0
45541,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
X=df.iloc[:,1:].values
y=df.iloc[:,0].values


In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
X_train, X_test,y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
X_train =X_train/255.0
X_test = X_test /255.0

In [ ]:
class customdataset(Dataset):
  def __init__(self,features,labels):
    self.features = torch.tensor(features, dtype =torch.float32)
    self.lables = torch.tensor(labels, dtype = torch.long)
  def __len__(self):
    return self.features.shape[0]
  def __getitem__(self, index):
    return self.features[index], self.lables[index]


In [ ]:
train_dataset = customdataset(X_train,y_train)
test_dataset = customdataset(X_test, y_test)

In [ ]:
train_dataloader = DataLoader(train_dataset,batch_size=32, shuffle=True, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle= False, pin_memory=True)

In [ ]:
from torch.nn.modules.activation import ReLU
class model(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.architecture = nn.Sequential(
        nn.Linear(num_features,128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128,64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(64, 10)
    )
  def forward(self,x):
    return self.architecture(x)


In [ ]:
epochs = 100
learning_rate =0.1

In [ ]:
model = model(X_train.shape[1])
model =model.to(device)

In [ ]:
optimiser = optim.SGD(model.parameters(),lr=learning_rate,weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

In [ ]:
for epoch in range(epochs):
  tot_epoch_loss = 0
  for batch_features, batch_labels in train_dataloader:
    batch_features, batch_labels =batch_features.to(device), batch_labels.to(device)
    outputs = model(batch_features)
    loss = criterion(outputs,batch_labels)
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
    tot_epoch_loss+=loss.item()
  tot_avg_loss = tot_epoch_loss/len(train_dataloader)
  print(f'Epoch:{epoch+1}, Loss:{tot_avg_loss}')



Epoch:1, Loss:0.6313204665184021
Epoch:2, Loss:0.49903353546063106
Epoch:3, Loss:0.4574561578929424
Epoch:4, Loss:0.4415320569376151
Epoch:5, Loss:0.41982606089611846
Epoch:6, Loss:0.4071724629799525
Epoch:7, Loss:0.39552438695232073
Epoch:8, Loss:0.3872691040933132
Epoch:9, Loss:0.38130253297090533
Epoch:10, Loss:0.3696552923818429
Epoch:11, Loss:0.3649938813497623
Epoch:12, Loss:0.36281574904421965
Epoch:13, Loss:0.35426111350953576
Epoch:14, Loss:0.34893572228898606
Epoch:15, Loss:0.3497027162909508
Epoch:16, Loss:0.3421484111795823
Epoch:17, Loss:0.3417572403301795
Epoch:18, Loss:0.33497465374817453
Epoch:19, Loss:0.334732779165109
Epoch:20, Loss:0.3315051493942738
Epoch:21, Loss:0.32880037898321945
Epoch:22, Loss:0.32412533748646577
Epoch:23, Loss:0.3231053833017747
Epoch:24, Loss:0.3202593529721101
Epoch:25, Loss:0.3214226141770681
Epoch:26, Loss:0.31655784051368635
Epoch:27, Loss:0.31565894426157076
Epoch:28, Loss:0.3167803720061978
Epoch:29, Loss:0.30991782883554697
Epoch:30, L

In [ ]:
model.eval()

model(
  (architecture): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [ ]:
correct =0
total = 0
with torch.no_grad():
  for batch_features, batch_labels in test_dataloader:
    batch_features, batch_labels =batch_features.to(device), batch_labels.to(device)
    outputs = model(batch_features)
    _,index  = torch.max(outputs, 1)
    total+= batch_labels.shape[0]
    correct += (index==batch_labels).sum().item()

print(correct/total)

0.89375


In [ ]:
correct =0
total = 0
with torch.no_grad():
  for batch_features, batch_labels in train_dataloader:
    batch_features, batch_labels =batch_features.to(device), batch_labels.to(device)
    outputs = model(batch_features)
    _,index  = torch.max(outputs, 1)
    total+= batch_labels.shape[0]
    correct += (index==batch_labels).sum().item()

print(correct/total)

0.9393333333333334
